# Telco Customer Churn Analysis

## Project Overview
This notebook analyzes the **Telco Customer Churn** dataset to understand customer behavior and identify key factors that lead to customer churn. The goal is to provide actionable insights that can help the company reduce churn rates.

### Dataset Description
The dataset contains information about a telecommunications company's customers, including:
- **Demographics**: Gender, Senior Citizen status, Partner, Dependents
- **Services**: Phone, Internet, Online Security, Streaming, Tech Support, etc.
- **Account Info**: Contract type, Payment method, Monthly/Total charges, Tenure
- **Target Variable**: Churn (Yes/No)

### Analysis Outline
1. Import Libraries
2. Load and Explore the Data
3. Data Cleaning and Preprocessing
4. Exploratory Data Analysis (EDA)
   - Univariate Analysis
   - Bivariate Analysis
   - Correlation Analysis
5. Key Insights and Recommendations

---
## 1. Import Libraries

In [ ]:
# Import essential libraries for data analysis and visualization
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Set visualization style and defaults
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

# Set pandas display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print('All libraries imported successfully!')

---
## 2. Load and Explore the Data

In [ ]:
# Load the dataset
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

# Display the first 5 rows
print('=== First 5 Rows ===')
df.head()

In [ ]:
# Check the shape of the dataset
print(f'Dataset Shape: {df.shape[0]} rows x {df.shape[1]} columns')
print(f'Total data points: {df.shape[0] * df.shape[1]:,}')

In [ ]:
# Display detailed info about the dataset
print('=== Dataset Info ===')
df.info()

In [ ]:
# Check unique values for each column
print('=== Unique Values Per Column ===')
for col in df.columns:
    unique_count = df[col].nunique()
    print(f'{col:<20s} : {unique_count:>5d} unique values')
    if unique_count <= 6:
        print(f'{"":<20s}   Values: {list(df[col].unique())}')

In [ ]:
# Statistical summary for numerical columns
print('=== Statistical Summary (Numerical) ===')
df.describe()

In [ ]:
# Statistical summary for categorical columns
print('=== Statistical Summary (Categorical) ===')
df.describe(include='object')

---
## 3. Data Cleaning and Preprocessing

In [ ]:
# Check for missing values
print('=== Missing Values ===')
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0])
if missing.sum() == 0:
    print('No null values detected. Checking for hidden missing values...')

In [ ]:
# Check TotalCharges for hidden missing values (spaces instead of NaN)
print('=== Checking TotalCharges for hidden missing values ===')
empty_total = df[df['TotalCharges'] == ' ']
print(f'Rows with empty TotalCharges: {len(empty_total)}')
print('\nThese rows (new customers with tenure=0):')
empty_total[['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']]

In [ ]:
# Fix TotalCharges: replace empty strings with NaN, convert to numeric, fill with 0
df['TotalCharges'] = df['TotalCharges'].replace(' ', np.nan)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)

# Convert SeniorCitizen from 0/1 to No/Yes for consistency
df['SeniorCitizen'] = df['SeniorCitizen'].map({0: 'No', 1: 'Yes'})

# Check for duplicates
print(f'Duplicate rows: {df.duplicated().sum()}')
print(f'Duplicate IDs:  {df["customerID"].duplicated().sum()}')
print(f'\nTotalCharges dtype: {df["TotalCharges"].dtype}')
print(f'Remaining nulls:    {df["TotalCharges"].isnull().sum()}')
print('\nData cleaning complete!')

In [ ]:
# Drop customerID (not useful for analysis) and create working copy
df_clean = df.drop(columns=['customerID'])

# Separate columns into numerical and categorical
numerical_cols = df_clean.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = df_clean.select_dtypes(include=['object']).columns.tolist()

print(f'Numerical columns ({len(numerical_cols)}): {numerical_cols}')
print(f'Categorical columns ({len(categorical_cols)}): {categorical_cols}')

In [ ]:
# Create a tenure group column for better analysis
def tenure_group(tenure):
    """Categorize tenure into groups."""
    if tenure <= 12: return '0-12 Months'
    elif tenure <= 24: return '13-24 Months'
    elif tenure <= 36: return '25-36 Months'
    elif tenure <= 48: return '37-48 Months'
    elif tenure <= 60: return '49-60 Months'
    else: return '61-72 Months'

df_clean['TenureGroup'] = df_clean['tenure'].apply(tenure_group)
print('=== Tenure Groups ===')
print(df_clean['TenureGroup'].value_counts().sort_index())

# Final cleaned dataset
print(f'\nCleaned dataset shape: {df_clean.shape}')
df_clean.head()

---
## 4. Exploratory Data Analysis (EDA)

### 4.1 Target Variable Analysis (Churn Distribution)

In [ ]:
# Churn distribution - count and percentage
churn_counts = df_clean['Churn'].value_counts()
churn_pct = df_clean['Churn'].value_counts(normalize=True) * 100

print('=== Churn Distribution ===')
for label in churn_counts.index:
    print(f'{label}: {churn_counts[label]:,} customers ({churn_pct[label]:.1f}%)')

# Visualize churn distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#2ecc71', '#e74c3c']

# Bar chart
bars = axes[0].bar(churn_counts.index, churn_counts.values, color=colors, edgecolor='black', width=0.5)
axes[0].set_title('Churn Count Distribution', fontweight='bold')
axes[0].set_xlabel('Churn')
axes[0].set_ylabel('Number of Customers')
for bar, count, pct in zip(bars, churn_counts.values, churn_pct.values):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 50,
                 f'{count:,}\n({pct:.1f}%)', ha='center', va='bottom', fontweight='bold')

# Pie chart
axes[1].pie(churn_counts.values, labels=churn_counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90, explode=(0, 0.05), textprops={'fontsize': 12})
axes[1].set_title('Churn Percentage', fontweight='bold')

plt.tight_layout()
plt.savefig('01_churn_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nKey Insight: ~26.5% of customers have churned (imbalanced dataset).')

### 4.2 Univariate Analysis - Numerical Features

In [ ]:
# Distribution of numerical columns with histograms and box plots
num_features = ['tenure', 'MonthlyCharges', 'TotalCharges']
colors_num = ['#3498db', '#e67e22', '#9b59b6']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Row 1: Histograms
for i, (col, color) in enumerate(zip(num_features, colors_num)):
    ax = axes[0, i]
    ax.hist(df_clean[col], bins=30, color=color, edgecolor='black', alpha=0.7)
    ax.set_title(f'Distribution of {col}', fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Frequency')
    ax.axvline(df_clean[col].mean(), color='red', linestyle='--', lw=2, label=f'Mean: {df_clean[col].mean():.1f}')
    ax.axvline(df_clean[col].median(), color='green', linestyle='-.', lw=2, label=f'Median: {df_clean[col].median():.1f}')
    ax.legend(fontsize=9)

# Row 2: Box plots
for i, (col, color) in enumerate(zip(num_features, colors_num)):
    ax = axes[1, i]
    bp = ax.boxplot(df_clean[col].dropna(), patch_artist=True,
                    boxprops=dict(facecolor=color, alpha=0.6),
                    medianprops=dict(color='red', linewidth=2))
    ax.set_title(f'Box Plot of {col}', fontweight='bold')
    ax.set_ylabel(col)

plt.tight_layout()
plt.savefig('02_numerical_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Detailed statistics for numerical features
print('=== Detailed Statistics ===')
for col in num_features:
    print(f'\n--- {col} ---')
    print(f'  Mean: {df_clean[col].mean():.2f} | Median: {df_clean[col].median():.2f}')
    print(f'  Std:  {df_clean[col].std():.2f} | Range: [{df_clean[col].min():.2f}, {df_clean[col].max():.2f}]')
    print(f'  Skew: {df_clean[col].skew():.2f} | Kurtosis: {df_clean[col].kurtosis():.2f}')

### 4.3 Univariate Analysis - Categorical Features

In [ ]:
# Distribution of demographic features
demo_cols = ['gender', 'SeniorCitizen', 'Partner', 'Dependents']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()
palette = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']

for i, col in enumerate(demo_cols):
    counts = df_clean[col].value_counts()
    bars = axes[i].bar(counts.index, counts.values, color=palette, edgecolor='black', width=0.5)
    axes[i].set_title(f'Distribution of {col}', fontweight='bold')
    axes[i].set_ylabel('Count')
    for bar in bars:
        h = bar.get_height()
        axes[i].text(bar.get_x() + bar.get_width()/2., h + 30, f'{int(h):,}',
                     ha='center', va='bottom', fontweight='bold')

plt.suptitle('Customer Demographics', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('03_demographics.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Distribution of service-related features
service_cols = ['PhoneService', 'MultipleLines', 'InternetService',
                'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                'TechSupport', 'StreamingTV', 'StreamingMovies']

fig, axes = plt.subplots(3, 3, figsize=(18, 14))
axes = axes.flatten()

for i, col in enumerate(service_cols):
    counts = df_clean[col].value_counts()
    axes[i].bar(counts.index, counts.values, color=['#3498db', '#e74c3c', '#2ecc71'],
                edgecolor='black', width=0.5)
    axes[i].set_title(f'{col}', fontweight='bold')
    axes[i].set_ylabel('Count')
    axes[i].tick_params(axis='x', rotation=30)

plt.suptitle('Services Distribution', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('04_services_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Distribution of account-related features
account_cols = ['Contract', 'PaperlessBilling', 'PaymentMethod']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, col in enumerate(account_cols):
    counts = df_clean[col].value_counts()
    bars = axes[i].bar(range(len(counts)), counts.values,
                       color=sns.color_palette('Set2', len(counts)), edgecolor='black', width=0.5)
    axes[i].set_title(f'{col}', fontweight='bold')
    axes[i].set_xticks(range(len(counts)))
    axes[i].set_xticklabels(counts.index, rotation=25, ha='right')
    axes[i].set_ylabel('Count')
    for bar in bars:
        h = bar.get_height()
        axes[i].text(bar.get_x() + bar.get_width()/2., h + 20, f'{int(h):,}',
                     ha='center', va='bottom', fontweight='bold')

plt.suptitle('Account Information', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('05_account_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.4 Bivariate Analysis - Churn vs Demographics

In [ ]:
# Churn rate by demographic features
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(demo_cols):
    churn_rate = df_clean.groupby(col)['Churn'].value_counts(normalize=True).unstack()
    churn_rate.plot(kind='bar', ax=axes[i], color=['#2ecc71', '#e74c3c'], edgecolor='black', width=0.6)
    axes[i].set_title(f'Churn Rate by {col}', fontweight='bold')
    axes[i].set_ylabel('Proportion')
    axes[i].set_xticklabels(axes[i].get_xticklabels(), rotation=0)
    axes[i].legend(title='Churn', loc='upper right')

plt.suptitle('Churn Rate by Demographics', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('06_churn_by_demographics.png', dpi=150, bbox_inches='tight')
plt.show()

# Print churn rates
print('=== Churn Rates by Demographics ===')
for col in demo_cols:
    print(f'\n--- {col} ---')
    churn_cross = pd.crosstab(df_clean[col], df_clean['Churn'], normalize='index') * 100
    for idx in churn_cross.index:
        print(f'  {idx}: Churn={churn_cross.loc[idx, "Yes"]:.1f}%')

### 4.5 Bivariate Analysis - Churn vs Services

In [ ]:
# Churn rate by service features
fig, axes = plt.subplots(3, 3, figsize=(20, 16))
axes = axes.flatten()

for i, col in enumerate(service_cols):
    churn_rate = df_clean.groupby(col)['Churn'].value_counts(normalize=True).unstack()
    churn_rate.plot(kind='bar', ax=axes[i], color=['#2ecc71', '#e74c3c'], edgecolor='black', width=0.6)
    axes[i].set_title(f'Churn by {col}', fontweight='bold', fontsize=11)
    axes[i].set_ylabel('Proportion')
    axes[i].set_xticklabels(axes[i].get_xticklabels(), rotation=30, ha='right')
    axes[i].legend(title='Churn', fontsize=8)

plt.suptitle('Churn Rate by Services', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('07_churn_by_services.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Deep dive: Churn by Internet Service type
fig, ax = plt.subplots(figsize=(10, 6))
internet_churn = pd.crosstab(df_clean['InternetService'], df_clean['Churn'])
internet_churn.plot(kind='bar', ax=ax, color=['#2ecc71', '#e74c3c'], edgecolor='black', width=0.6)
ax.set_title('Customer Count by Internet Service and Churn', fontweight='bold')
ax.set_xlabel('Internet Service Type')
ax.set_ylabel('Number of Customers')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
for container in ax.containers:
    ax.bar_label(container, fmt='%d', label_type='edge', fontweight='bold')

plt.tight_layout()
plt.savefig('08_churn_by_internet.png', dpi=150, bbox_inches='tight')
plt.show()

print('=== Churn Rate by Internet Service ===')
print((pd.crosstab(df_clean['InternetService'], df_clean['Churn'], normalize='index') * 100).round(1))

### 4.6 Bivariate Analysis - Churn vs Account Info

In [ ]:
# Churn by account features
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, col in enumerate(account_cols):
    churn_rate = pd.crosstab(df_clean[col], df_clean['Churn'], normalize='index') * 100
    churn_rate.plot(kind='bar', ax=axes[i], color=['#2ecc71', '#e74c3c'], edgecolor='black', width=0.6)
    axes[i].set_title(f'Churn Rate by {col}', fontweight='bold')
    axes[i].set_ylabel('Percentage (%)')
    axes[i].set_xticklabels(axes[i].get_xticklabels(), rotation=25, ha='right')
    for container in axes[i].containers:
        axes[i].bar_label(container, fmt='%.1f%%', label_type='center', fontsize=9, fontweight='bold')

plt.suptitle('Churn Rate by Account Information', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('09_churn_by_account.png', dpi=150, bbox_inches='tight')
plt.show()

print('Key Insight: Month-to-month contracts: ~42.7% churn')
print('Key Insight: Electronic check: ~45.3% churn')

### 4.7 Bivariate Analysis - Churn vs Numerical Features

In [ ]:
# Box plots and KDE plots: Numerical features by Churn
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Row 1: Box plots
for i, col in enumerate(num_features):
    sns.boxplot(x='Churn', y=col, data=df_clean, ax=axes[0, i],
                palette={'No': '#2ecc71', 'Yes': '#e74c3c'}, width=0.5)
    axes[0, i].set_title(f'{col} by Churn', fontweight='bold')

# Row 2: KDE plots
for i, col in enumerate(num_features):
    sns.kdeplot(data=df_clean, x=col, hue='Churn', ax=axes[1, i],
                palette={'No': '#2ecc71', 'Yes': '#e74c3c'},
                fill=True, alpha=0.3, linewidth=2)
    axes[1, i].set_title(f'{col} Density by Churn', fontweight='bold')

plt.tight_layout()
plt.savefig('10_numerical_by_churn.png', dpi=150, bbox_inches='tight')
plt.show()

# Compare mean values
print('=== Mean Values by Churn Status ===')
print(df_clean.groupby('Churn')[num_features].mean().round(2))
print('\nInsight: Churned customers have lower tenure and higher monthly charges.')

### 4.8 Churn by Tenure Groups

In [ ]:
# Churn rate by tenure group
tenure_order = ['0-12 Months', '13-24 Months', '25-36 Months',
                '37-48 Months', '49-60 Months', '61-72 Months']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Count plot
sns.countplot(x='TenureGroup', hue='Churn', data=df_clean, order=tenure_order,
              palette={'No': '#2ecc71', 'Yes': '#e74c3c'}, edgecolor='black', ax=axes[0])
axes[0].set_title('Customer Count by Tenure Group', fontweight='bold')
axes[0].tick_params(axis='x', rotation=30)

# Churn rate plot
tenure_churn = df_clean.groupby('TenureGroup')['Churn'].apply(
    lambda x: (x == 'Yes').sum() / len(x) * 100).reindex(tenure_order)
bars = axes[1].bar(tenure_order, tenure_churn.values,
                   color=sns.color_palette('YlOrRd', len(tenure_order)), edgecolor='black', width=0.6)
axes[1].set_title('Churn Rate (%) by Tenure Group', fontweight='bold')
axes[1].set_ylabel('Churn Rate (%)')
axes[1].tick_params(axis='x', rotation=30)
for bar in bars:
    h = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2., h + 0.5,
                 f'{h:.1f}%', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('11_churn_by_tenure.png', dpi=150, bbox_inches='tight')
plt.show()
print('Key Insight: New customers (0-12 months) have the highest churn rate!')

### 4.9 Correlation Analysis

In [ ]:
# Encode binary columns for correlation analysis
df_corr = df_clean.copy()
df_corr['Churn_Binary'] = (df_corr['Churn'] == 'Yes').astype(int)

binary_map = {'gender': 'Male', 'SeniorCitizen': 'Yes', 'Partner': 'Yes',
              'Dependents': 'Yes', 'PhoneService': 'Yes', 'PaperlessBilling': 'Yes'}
for col, yes_val in binary_map.items():
    df_corr[col + '_Enc'] = (df_corr[col] == yes_val).astype(int)

# Correlation matrix
corr_cols = ['tenure', 'MonthlyCharges', 'TotalCharges',
             'gender_Enc', 'SeniorCitizen_Enc', 'Partner_Enc',
             'Dependents_Enc', 'PaperlessBilling_Enc', 'Churn_Binary']
corr_matrix = df_corr[corr_cols].corr()

# Heatmap
fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, mask=mask, square=True, linewidths=0.5, ax=ax)
ax.set_title('Correlation Matrix Heatmap', fontweight='bold', fontsize=14)

plt.tight_layout()
plt.savefig('12_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Features most correlated with Churn
print('=== Correlation with Churn (sorted) ===')
churn_corr = corr_matrix['Churn_Binary'].drop('Churn_Binary').sort_values(key=abs, ascending=False)
for feat, val in churn_corr.items():
    print(f'{feat:<25s}: {val:>7.3f} ({"Positive" if val > 0 else "Negative"})')

In [ ]:
# Scatter plots: key relationships colored by Churn
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Monthly vs Total Charges
for churn_val, color, label in [('No', '#2ecc71', 'No Churn'), ('Yes', '#e74c3c', 'Churn')]:
    mask = df_clean['Churn'] == churn_val
    axes[0].scatter(df_clean.loc[mask, 'MonthlyCharges'], df_clean.loc[mask, 'TotalCharges'],
                    alpha=0.3, c=color, label=label, s=15)
axes[0].set_title('Monthly vs Total Charges by Churn', fontweight='bold')
axes[0].set_xlabel('Monthly Charges ($)')
axes[0].set_ylabel('Total Charges ($)')
axes[0].legend(markerscale=3)

# Tenure vs Monthly Charges
for churn_val, color, label in [('No', '#2ecc71', 'No Churn'), ('Yes', '#e74c3c', 'Churn')]:
    mask = df_clean['Churn'] == churn_val
    axes[1].scatter(df_clean.loc[mask, 'tenure'], df_clean.loc[mask, 'MonthlyCharges'],
                    alpha=0.3, c=color, label=label, s=15)
axes[1].set_title('Tenure vs Monthly Charges by Churn', fontweight='bold')
axes[1].set_xlabel('Tenure (Months)')
axes[1].set_ylabel('Monthly Charges ($)')
axes[1].legend(markerscale=3)

plt.tight_layout()
plt.savefig('13_scatter_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Insight: Churned customers cluster at high monthly charges and low tenure.')

### 4.10 Advanced Visualizations

In [ ]:
# Violin plot: Monthly Charges by Contract Type and Churn
fig, ax = plt.subplots(figsize=(12, 6))
sns.violinplot(x='Contract', y='MonthlyCharges', hue='Churn', data=df_clean,
               palette={'No': '#2ecc71', 'Yes': '#e74c3c'}, split=True, inner='quartile', ax=ax)
ax.set_title('Monthly Charges by Contract Type and Churn', fontweight='bold')
ax.set_xlabel('Contract Type')
ax.set_ylabel('Monthly Charges ($)')
plt.tight_layout()
plt.savefig('14_violin_contract.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Heatmap: Churn rate by Contract Type and Payment Method
fig, ax = plt.subplots(figsize=(10, 6))
churn_pivot = df_clean.pivot_table(values='Churn', index='Contract', columns='PaymentMethod',
                                    aggfunc=lambda x: (x == 'Yes').sum() / len(x) * 100)
sns.heatmap(churn_pivot, annot=True, fmt='.1f', cmap='YlOrRd',
            linewidths=1, cbar_kws={'label': 'Churn Rate (%)'}, ax=ax)
ax.set_title('Churn Rate by Contract x Payment Method', fontweight='bold')
plt.tight_layout()
plt.savefig('15_heatmap_contract_payment.png', dpi=150, bbox_inches='tight')
plt.show()
print('Key Insight: Month-to-month + Electronic check = highest churn rate!')

In [ ]:
# Number of additional services vs Churn
additional_services = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                       'TechSupport', 'StreamingTV', 'StreamingMovies']
df_clean['NumServices'] = df_clean[additional_services].apply(
    lambda row: sum(1 for val in row if val == 'Yes'), axis=1)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Churn rate by number of services
svc_churn = df_clean.groupby('NumServices')['Churn'].apply(
    lambda x: (x == 'Yes').sum() / len(x) * 100)
bars = axes[0].bar(svc_churn.index, svc_churn.values,
                   color=sns.color_palette('RdYlGn_r', len(svc_churn)), edgecolor='black', width=0.6)
axes[0].set_title('Churn Rate by # of Services', fontweight='bold')
axes[0].set_xlabel('Number of Additional Services')
axes[0].set_ylabel('Churn Rate (%)')
for bar in bars:
    h = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2., h + 0.5,
                 f'{h:.1f}%', ha='center', fontweight='bold')

# Customer count by number of services
svc_count = df_clean['NumServices'].value_counts().sort_index()
axes[1].bar(svc_count.index, svc_count.values,
            color=sns.color_palette('Blues', len(svc_count)), edgecolor='black', width=0.6)
axes[1].set_title('Customer Count by # of Services', fontweight='bold')
axes[1].set_xlabel('Number of Additional Services')
axes[1].set_ylabel('Customers')

plt.tight_layout()
plt.savefig('16_services_count.png', dpi=150, bbox_inches='tight')
plt.show()
print('Insight: Fewer services = higher churn. Bundling services reduces churn.')

### 4.11 Revenue Impact Analysis

In [ ]:
# Revenue impact of churn
churned = df_clean[df_clean['Churn'] == 'Yes']
stayed = df_clean[df_clean['Churn'] == 'No']

print('=== Revenue Impact Analysis ===')
print(f'Total Revenue:      ${df_clean["TotalCharges"].sum():>12,.2f}')
print(f'From Churned:       ${churned["TotalCharges"].sum():>12,.2f}')
print(f'From Retained:      ${stayed["TotalCharges"].sum():>12,.2f}')
print(f'\nMonthly At Risk:    ${churned["MonthlyCharges"].sum():>12,.2f}')
print(f'Annual At Risk:     ${churned["MonthlyCharges"].sum() * 12:>12,.2f}')
print(f'\nAvg Monthly (Churned):  ${churned["MonthlyCharges"].mean():.2f}')
print(f'Avg Monthly (Stayed):   ${stayed["MonthlyCharges"].mean():.2f}')

# Revenue visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
labels = ['Retained', 'Churned']
colors = ['#2ecc71', '#e74c3c']

# Total monthly revenue
rev_data = [stayed['MonthlyCharges'].sum(), churned['MonthlyCharges'].sum()]
bars = axes[0].bar(labels, rev_data, color=colors, edgecolor='black', width=0.5)
axes[0].set_title('Total Monthly Revenue by Status', fontweight='bold')
axes[0].set_ylabel('Monthly Revenue ($)')
for bar in bars:
    h = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2., h + 500,
                 f'${h:,.0f}', ha='center', fontweight='bold')

# Average monthly charges
avg_data = [stayed['MonthlyCharges'].mean(), churned['MonthlyCharges'].mean()]
bars2 = axes[1].bar(labels, avg_data, color=colors, edgecolor='black', width=0.5)
axes[1].set_title('Avg Monthly Charges by Status', fontweight='bold')
axes[1].set_ylabel('Avg Monthly Charges ($)')
for bar in bars2:
    h = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2., h + 0.5,
                 f'${h:.2f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('17_revenue_impact.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.12 Summary Dashboard

In [ ]:
# Comprehensive summary dashboard
fig, axes = plt.subplots(2, 3, figsize=(20, 12))

# 1. Churn pie
churn_vals = df_clean['Churn'].value_counts()
axes[0,0].pie(churn_vals, labels=churn_vals.index, autopct='%1.1f%%',
              colors=['#2ecc71', '#e74c3c'], startangle=90, explode=(0, 0.05))
axes[0,0].set_title('Overall Churn Rate', fontweight='bold')

# 2. Churn by Contract
ct = pd.crosstab(df_clean['Contract'], df_clean['Churn'], normalize='index') * 100
ct['Yes'].plot(kind='bar', ax=axes[0,1], color=['#e74c3c','#f39c12','#2ecc71'], edgecolor='black', width=0.5)
axes[0,1].set_title('Churn % by Contract', fontweight='bold')
axes[0,1].set_ylabel('Churn Rate (%)')
axes[0,1].tick_params(axis='x', rotation=30)

# 3. Churn by Internet
ci = pd.crosstab(df_clean['InternetService'], df_clean['Churn'], normalize='index') * 100
ci['Yes'].plot(kind='bar', ax=axes[0,2], color=['#3498db','#e74c3c','#95a5a6'], edgecolor='black', width=0.5)
axes[0,2].set_title('Churn % by Internet', fontweight='bold')
axes[0,2].set_ylabel('Churn Rate (%)')
axes[0,2].tick_params(axis='x', rotation=30)

# 4. Tenure by Churn
sns.boxplot(x='Churn', y='tenure', data=df_clean, ax=axes[1,0],
            palette={'No': '#2ecc71', 'Yes': '#e74c3c'}, width=0.5)
axes[1,0].set_title('Tenure by Churn', fontweight='bold')

# 5. Monthly Charges by Churn
sns.boxplot(x='Churn', y='MonthlyCharges', data=df_clean, ax=axes[1,1],
            palette={'No': '#2ecc71', 'Yes': '#e74c3c'}, width=0.5)
axes[1,1].set_title('Monthly Charges by Churn', fontweight='bold')

# 6. Churn by Payment
cp = pd.crosstab(df_clean['PaymentMethod'], df_clean['Churn'], normalize='index') * 100
cp['Yes'].sort_values(ascending=True).plot(kind='barh', ax=axes[1,2], color='#e74c3c', edgecolor='black')
axes[1,2].set_title('Churn % by Payment', fontweight='bold')
axes[1,2].set_xlabel('Churn Rate (%)')

plt.suptitle('TELCO CUSTOMER CHURN - ANALYSIS DASHBOARD', fontsize=18, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('18_summary_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Key Insights and Recommendations

### Key Findings

#### 1. Overall Churn Rate
- **26.5%** of customers have churned — a significant rate for a telecom company.

#### 2. Contract Type is the Strongest Factor
- **Month-to-month**: ~42.7% churn | **One year**: ~11.3% | **Two year**: ~2.8%

#### 3. Internet Service Type Matters
- **Fiber optic** churn: ~41.9% vs **DSL**: ~19.0% — possible quality/pricing issue.

#### 4. Tenure is Critical
- New customers (0-12 months) churn at ~47.7%. Avg tenure: churned ~18 mo vs retained ~38 mo.

#### 5. Pricing Impact
- Churned customers pay more monthly ($74.44 vs $61.27).

#### 6. Payment Method
- **Electronic check** has ~45.3% churn. Automatic methods are much lower.

#### 7. Additional Services Reduce Churn
- Fewer services = higher churn. Bundling creates stickiness.

#### 8. Senior Citizens
- Senior citizens churn at ~41.7% vs non-seniors at ~23.6%.

### Business Recommendations

1. **Retain New Customers**: Implement onboarding programs for the first 12 months.
2. **Promote Long-term Contracts**: Offer incentives to switch from month-to-month plans.
3. **Review Fiber Optic Service**: Investigate pricing/quality issues causing high churn.
4. **Encourage Auto-Pay**: Offer discounts for automatic payment enrollment.
5. **Bundle Services**: Create attractive packages with security, backup, and support.
6. **Senior Loyalty Programs**: Develop targeted retention for senior citizens.
7. **Monitor High-Charge Customers**: Proactively engage customers with high monthly bills.

In [ ]:
# Final summary statistics
print('=' * 60)
print('       TELCO CUSTOMER CHURN - ANALYSIS SUMMARY')
print('=' * 60)
print(f'\nDataset: {df.shape[0]:,} customers | {df.shape[1]} features')
print(f'Overall Churn Rate: {(df_clean["Churn"] == "Yes").mean() * 100:.1f}%')
print(f'Churned:  {(df_clean["Churn"] == "Yes").sum():,}')
print(f'Retained: {(df_clean["Churn"] == "No").sum():,}')
print(f'\n--- Top Risk Factors ---')
print(f'  1. Month-to-month contract: ~42.7% churn')
print(f'  2. Fiber optic internet:    ~41.9% churn')
print(f'  3. Electronic check:        ~45.3% churn')
print(f'  4. New customers (0-12 mo): ~47.7% churn')
print(f'  5. Senior citizens:         ~41.7% churn')
print(f'\n--- Revenue at Risk ---')
print(f'  Monthly: ${churned["MonthlyCharges"].sum():,.2f}')
print(f'  Annual:  ${churned["MonthlyCharges"].sum() * 12:,.2f}')
print(f'\n{"=" * 60}')
print('Analysis completed successfully!')